# Foodvalley Hydrogen Siting — Pipeline Walkthrough

This notebook walks through the regional (PC4-level) hydrogen-hub siting pipeline
("MCA1") for Regio Foodvalley, reproduced in open-source Python from the raw
Liander/Stedin open datasets.

Background: Regio Foodvalley (eight Dutch municipalities) faces severe electricity
grid congestion. This analysis identifies which postal-code zones (PC4) have the
largest gap between local renewable electricity supply and demand, combined with
the most urgent grid congestion status — the best candidate zones for siting a
green-hydrogen conversion & storage hub that absorbs surplus renewable electricity.

Full methodology: see `../README.md` and `../CASE_STUDY.md`. Original report:
*"Relieving grid congestion with hydrogen"* (ACT Team 3.695, Wageningen University, 2026).

In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import matplotlib.pyplot as plt

from pipeline import build_pc4_table, score
from sensitivity import normalize, entropy_weights, critic_weights, exact_weight_stability_interval
import numpy as np

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

## 1. Load & assemble the PC4 table from raw data

This step parses the real Liander/Stedin small-connection open datasets (267,797 + 147,770 rows), applies the documented data-cleaning fixes (DSO-boundary-artefact exclusion, whitespace-stripped product filter), joins the small reference tables (renewables, industrial proxy, congestion status), and computes the "invisible factory"-corrected demand.

In [2]:
df = build_pc4_table()
print(f"{len(df)} PC4 zones assembled")
df[["PC4", "municipality", "demand_dso_gwh", "demand_synthetic_gwh", "total_supply_gwh", "congestion_status"]].head(10)

Excluded 0 DSO-boundary-artefact PC4 zones: ['3905', '3925', '3927']
73 PC4 zones assembled


,PC4,municipality,demand_dso_gwh,demand_synthetic_gwh,total_supply_gwh,congestion_status
0,3727,Renswoude,0.000000,0.000000,0.000000,GREEN
1,3771,Barneveld,34.139427,267.299427,17.594480,AMBER
2,3772,Barneveld,30.572945,31.469135,11.111407,AMBER
3,3775,Barneveld,1.111869,1.145569,0.374460,AMBER
4,3776,Barneveld,5.125461,5.600031,1.798365,AMBER
5,3781,Barneveld,24.532811,25.777511,8.708777,AMBER
6,3784,Barneveld,4.305706,4.531336,1.613313,AMBER
7,3785,Barneveld,3.535210,3.600350,1.402125,AMBER
8,3792,Barneveld,0.530205,0.534685,0.215555,AMBER
9,3794,Barneveld,1.723026,1.802566,0.669733,AMBER


## 2. Score: composite siting score = 50% energy deficit + 50% grid congestion urgency

In [3]:
scored = score(df)
top10 = scored[["PC4", "municipality", "business_park", "demand_synthetic_gwh", "net_balance_gwh", "congestion_status", "composite_siting"]].head(10)
top10

,PC4,municipality,business_park,demand_synthetic_gwh,net_balance_gwh,congestion_status,composite_siting
0,3771,Barneveld,Harselaar Barneveld,267.299427,-249.704947,AMBER,0.7500
1,3901,Veenendaal,Nijverkamp Veenendaal,84.096000,-84.096000,RED,0.6684
2,3903,Veenendaal,De Compagnie Veenendaal,46.032000,-46.032000,RED,0.5922
3,6718,Ede,BT A12/Schampsteeg Ede,50.660647,-43.155632,RED,0.5864
4,6711,Ede,NaN,29.804088,-24.890713,RED,0.5498
5,6717,Ede,NaN,27.663203,-20.582134,RED,0.5412
6,3881,Ede,NaN,26.326474,-20.186002,RED,0.5404
7,6721,Veenendaal,NaN,27.753488,-19.108352,RED,0.5383
8,6716,Ede,NaN,19.834038,-15.372608,RED,0.5308
9,3841,Ede,Harselaar (Ede side),16.227378,-14.017123,RED,0.5281


In [4]:
fig, ax = plt.subplots(figsize=(9, 5))
top = scored.head(10).iloc[::-1]
colors = top["congestion_status"].map({"RED": "#C0392B", "AMBER": "#D68910", "GREEN": "#27964D"})
ax.barh(top["PC4"] + " " + top["municipality"], top["composite_siting"], color=colors)
ax.set_xlabel("Composite siting score")
ax.set_title("Top 10 candidate PC4 zones — reproduced from raw data")
plt.tight_layout()
plt.show()

C:\Users\beswa\AppData\Local\Temp\ipykernel_20080\1747696498.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Cross-check against the canonical, submitted-report dataset

`data/processed/hydrogen_siting_phase3_v2.csv` is the exact dataset behind Table 4
of the submitted report. Comparing against it validates the from-scratch
reproduction above.

In [5]:
canonical = pd.read_csv("../data/processed/hydrogen_siting_phase3_v2.csv", dtype={"PC4": str})
canonical_top10 = canonical.sort_values("composite_siting", ascending=False)["PC4"].head(10).tolist()
reproduced_top10 = scored["PC4"].head(10).tolist()

overlap = len(set(canonical_top10) & set(reproduced_top10))
print(f"Top-10 overlap: {overlap}/10 PC4 zones")
print(f"Canonical  #1: {canonical.sort_values('composite_siting', ascending=False).iloc[0]['PC4']} "
      f"(score {canonical.sort_values('composite_siting', ascending=False).iloc[0]['composite_siting']:.4f})")
print(f"Reproduced #1: {scored.iloc[0]['PC4']} (score {scored.iloc[0]['composite_siting']:.4f})")

Top-10 overlap: 9/10 PC4 zones
Canonical  #1: 3771 (score 0.7500)
Reproduced #1: 3771 (score 0.7500)


## 4. Is the #1 recommendation robust to how the weights are chosen?

The submitted report uses a fixed 50/50 split between the two criteria but never
quantifies how sensitive the ranking is to that choice. Here we derive the exact
**Weight Stability Interval (WSI)**: the range of the deficit weight over which
the #1-ranked site does not change, plus two independent data-driven weighting
schemes (entropy, CRITIC) for comparison. Run on the canonical dataset so the
numbers line up with what was actually submitted.

In [6]:
s_deficit = normalize(-canonical["net_balance_p3_gwh"])
s_congestion = canonical["congestion_score"]
X = np.column_stack([s_deficit.values, s_congestion.values])

w_entropy = entropy_weights(X)
w_critic = critic_weights(X)
print("Entropy weights (deficit, congestion):", w_entropy.round(3))
print("CRITIC weights  (deficit, congestion):", w_critic.round(3))

target_pc4, w_low, w_high = exact_weight_stability_interval(X, canonical, w1_target=0.5)
print(f"\nPC4 {target_pc4} stays #1 for any deficit weight in [{w_low:.1%}, {w_high:.1%}]")

Entropy weights (deficit, congestion): [0.885 0.115]
CRITIC weights  (deficit, congestion): [0.276 0.724]

PC4 3771 stays #1 for any deficit weight in [47.0%, 100.0%]


**Interpretation**: the report's top recommendation (Harselaar, PC4 3771) is robust
to a wide range of reasonable reweighting (47%-100% deficit weight) — but *not*
robust to a congestion-dominant weighting scheme (e.g. the objective CRITIC
weighting, 28% deficit / 72% congestion), under which PC4 3901 (Nijverkamp,
Veenendaal — the only officially RED-congested top candidate) takes #1 instead.
See `figures/sensitivity_weight_stability.png` and the interactive
`dashboard/index.html` (drag the weight slider) to explore this directly.